# RSNA Knee: Wide-ResNet50 Specialist 5-Fold Training

Trained on top of the 0.954+ Clinical Specialist Architecture:
1. Native 3-slice positional windowing (8 windows x 3 consecutive slices = 24 slices per study)
2. 0.8 center field-of-view crop (`c08`) focusing high resolution on articular cartilage, joint space, and ligaments
3. Backbone: `wide_resnet50_2` with ImageNet initialization and exact tensor normalization
4. Multi-finding 12-head attention pooling (`RaptorClassifier`) allowing independent slice weighting per pathology
5. Logit clamping `[-12.0, 12.0]` + Gradient Clipping `max_norm=1.0` for rock-solid FP16 AMP stability
6. Clinical Specialist Focal Loss: Hyper-weighted for Lateral OA (3.0), Baker's (2.2), Cruciate Ligaments (2.0), and Contusion (1.8)
7. Step 1: Full-Corpus pre-training on all 4,407 studies with soft pseudo-labels to warm up attention head weights
8. Step 2: Stratified 5-Fold fine-tuning on ground truth folds with cosine annealing learning rate scheduling.

In [ ]:
import os
import glob
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
import timm
from tqdm import tqdm

def resolve_dir(target_name):
    for root, dirs, files in os.walk("/kaggle/input"):
        if target_name in dirs:
            return os.path.join(root, target_name)
    for pat in [f"/kaggle/input/{target_name}", f"/kaggle/input/datasets/*/{target_name}", f"/kaggle/input/*/{target_name}"]:
        matches = glob.glob(pat)
        if matches:
            return matches[0]
    return f"/kaggle/input/datasets/barun2104/{target_name}"

def resolve_file(target_file):
    for root, dirs, files in os.walk("/kaggle/input"):
        if target_file in files:
            return os.path.join(root, target_file)
    for pat in [f"/kaggle/input/**/{target_file}", f"/kaggle/input/*/*/{target_file}"]:
        matches = glob.glob(pat, recursive=True)
        if matches:
            return matches[0]
    return f"/kaggle/input/competitions/rsna-knee-abnormality-detection/{target_file}"

IMG_DIR = resolve_dir("rsna-knee-mri-processed-3d-volumes")
CSV_PATH = resolve_file("train_folds_with_pseudo.csv")
TRAIN_SERIES_PATH = resolve_file("train_series.csv")

TARGET_COLS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]
print(f"IMG_DIR: {IMG_DIR}")
print(f"CSV_PATH: {CSV_PATH}")
print(f"TRAIN_SERIES_PATH: {TRAIN_SERIES_PATH}")


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using compute device:", device)
df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} total studies with folds and pseudo-labels.")
df[TARGET_COLS] = df[TARGET_COLS].fillna(0.5)
series_df = pd.read_csv(TRAIN_SERIES_PATH)
study_to_series_map = series_df.groupby("StudyInstanceUID")["SeriesInstanceUID"].apply(list).to_dict()
print(f"Mapped {len(study_to_series_map)} studies to series volumes.")


In [ ]:
def _crop(t, ratio=0.8):
    h, w = t.shape[-2:]
    ch, cw = int(h * ratio), int(w * ratio)
    y0 = (h - ch) // 2
    x0 = (w - cw) // 2
    return t[..., y0:y0 + ch, x0:x0 + cw]

_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

class WindowedKneeDataset(Dataset):
    def __init__(self, df, series_map, img_dir=IMG_DIR, is_train=True):
        self.df = df.reset_index(drop=True)
        self.series_map = series_map
        self.img_dir = img_dir
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        study_id = row["StudyInstanceUID"]
        series_ids = self.series_map.get(study_id, [])
        loaded_vols = []
        for s_id in series_ids:
            p = os.path.join(self.img_dir, f"{s_id}.npz")
            if os.path.exists(p):
                try:
                    with np.load(p) as npz:
                        k = "data" if "data" in npz else npz.files[0]
                        loaded_vols.append(npz[k])
                except Exception:
                    pass
        if len(loaded_vols) == 0:
            vol = np.zeros((24, 224, 224), dtype=np.float32)
        else:
            vol = np.concatenate(loaded_vols, axis=0)
            if vol.shape[0] != 24:
                idx_slices = np.linspace(0, vol.shape[0] - 1, 24).astype(int)
                vol = vol[idx_slices]
        
        windows = vol.reshape(8, 3, vol.shape[1], vol.shape[2]).astype(np.float32)
        if windows.max() > 1.0:
            windows /= 255.0
        tensor_w = torch.from_numpy(windows)
        
        tensor_w = _crop(tensor_w)
        
        tensor_w = F.interpolate(tensor_w, size=(224, 224), mode="bilinear", align_corners=False)
        tensor_w = (tensor_w - _MEAN) / _STD
        
        if self.is_train and np.random.rand() > 0.5:
            tensor_w = torch.flip(tensor_w, dims=[-1])
        y_vals = np.nan_to_num(row[TARGET_COLS].values.astype(np.float32), nan=0.5)
        y = torch.tensor(y_vals)
        return tensor_w, y


In [ ]:
class RaptorClassifier(nn.Module):
    def __init__(self, backbone, F_dim, n=12, drop=0.2):
        super().__init__()
        self.backbone = backbone
        self.norm = nn.LayerNorm(F_dim)
        self.att = nn.Sequential(
            nn.Linear(F_dim, 256),
            nn.Tanh(),
            nn.Dropout(drop),
            nn.Linear(256, n)
        )
        self.clsW = nn.Parameter(torch.zeros(n, F_dim))
        self.clsb = nn.Parameter(torch.zeros(n))
        nn.init.trunc_normal_(self.clsW, std=0.02)
        self.n = n

    def encode(self, x):
        B, K = x.shape[:2]
        flat = x.flatten(0, 1) # (B*K, 3, 224, 224)
        f = torch.cat([self.backbone(chunk) for chunk in flat.split(8)], dim=0)
        return f.view(B, K, -1)

    def head(self, feats):
        h = self.norm(feats)
        a = self.att(h)
        a = torch.softmax(a, dim=1)
        pooled = torch.einsum("bkn,bkf->bnf", a, h)
        logits = (pooled * self.clsW).sum(-1) + self.clsb
        return torch.clamp(logits, -12.0, 12.0)

    def forward(self, x):
        return self.head(self.encode(x))

def build_model(pretrained=True):
    bb = timm.create_model("wide_resnet50_2", pretrained=pretrained, num_classes=0, in_chans=3)
    return RaptorClassifier(bb, F_dim=bb.num_features, n=len(TARGET_COLS))


In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, weights, gamma=1.5):
        super().__init__()
        self.register_buffer("w", torch.tensor(weights, dtype=torch.float32))
        self.gamma = gamma
    def forward(self, logits, targets):
        logits = torch.clamp(logits, -12.0, 12.0)
        targets = torch.clamp(targets, 1e-4, 1.0 - 1e-4)
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        p = torch.sigmoid(logits)
        pt = targets * p + (1.0 - targets) * (1.0 - p)
        focal_weight = ((1.0 - pt).clamp(min=1e-4, max=1.0) ** self.gamma) * self.w
        loss = (focal_weight * bce).mean()
        return loss if torch.isfinite(loss) else bce.mean()

WEIGHTS = [2.0, 2.0, 1.6, 1.6, 1.4, 3.0, 1.4, 1.6, 1.8, 2.2, 1.8, 1.5]
criterion = FocalLoss(WEIGHTS).to(device)

def compute_macro_auc(y_t, y_p):
    y_t = np.nan_to_num(y_t, nan=0.0)
    y_p = np.nan_to_num(y_p, nan=0.5)
    aucs = []
    for i in range(y_t.shape[1]):
        if len(np.unique(y_t[:, i])) > 1:
            try:
                aucs.append(roc_auc_score(y_t[:, i], y_p[:, i]))
            except Exception:
                aucs.append(0.5)
    return float(np.mean(aucs)) if aucs else 0.5


In [ ]:
print("=== Starting Wide-ResNet50 Specialist 5-Fold Training ===")
# Step 1: Pre-training on full dataset (4,407 studies) for 2 epochs to warm up attention head
pre_model = build_model(pretrained=True).to(device)
pre_loader = DataLoader(WindowedKneeDataset(df, study_to_series_map, is_train=True), batch_size=8, shuffle=True, num_workers=2)
pre_opt = torch.optim.AdamW(pre_model.parameters(), lr=5e-5, weight_decay=1e-2)
scaler = torch.amp.GradScaler("cuda")

for epoch in range(1, 3):
    pre_model.train()
    pbar = tqdm(pre_loader, desc=f"Pretrain Epoch {epoch}/2")
    total_loss = 0.0
    for x, y in pbar:
        x, y = x.to(device), y.to(device)
        pre_opt.zero_grad()
        with torch.amp.autocast("cuda"):
            out = pre_model(x)
            loss = criterion(out, y)
        scaler.scale(loss).backward()
        scaler.unscale_(pre_opt)
        torch.nn.utils.clip_grad_norm_(pre_model.parameters(), max_norm=1.0)
        scaler.step(pre_opt)
        scaler.update()
        total_loss += loss.item() if torch.isfinite(loss) else 0.0
        pbar.set_postfix({"loss": f"{total_loss / (pbar.n + 1):.4f}"})

torch.save(pre_model.state_dict(), "/kaggle/working/wideresnet50_specialist_pretrain.pt")
print("Pre-training completed and saved backbone weights.")


In [ ]:
# Step 2: 5-Fold Stratified Fine-tuning
fold_aucs = []
gt_df = df[df["fold"] >= 0].copy().reset_index(drop=True)

for fold in range(5):
    print(f"\n--- Fine-tuning Fold {fold}/5 ---")
    tr_df = gt_df[gt_df["fold"] != fold]
    va_df = gt_df[gt_df["fold"] == fold]
    
    tr_loader = DataLoader(WindowedKneeDataset(tr_df, study_to_series_map, is_train=True), batch_size=8, shuffle=True, num_workers=2)
    va_loader = DataLoader(WindowedKneeDataset(va_df, study_to_series_map, is_train=False), batch_size=8, shuffle=False, num_workers=2)
    
    model = build_model(pretrained=False).to(device)
    model.load_state_dict(torch.load("/kaggle/working/wideresnet50_specialist_pretrain.pt"))
    
    bb_params = [p for n, p in model.named_parameters() if "backbone" in n]
    hd_params = [p for n, p in model.named_parameters() if "backbone" not in n]
    opt = torch.optim.AdamW([
        {"params": bb_params, "lr": 1.5e-5},
        {"params": hd_params, "lr": 1.5e-4}
    ], weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=4, eta_min=1e-6)
    
    best_auc = 0.0
    best_path = f"/kaggle/working/wideresnet50_specialist_f{fold}.pt"
    
    for ep in range(1, 5):
        model.train()
        for x, y in tr_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            with torch.amp.autocast("cuda"):
                loss = criterion(model(x), y)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(opt)
            scaler.update()
        sched.step()
        
        model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for x, y in va_loader:
                x = x.to(device)
                with torch.amp.autocast("cuda"):
                    out = torch.sigmoid(model(x))
                preds.append(np.nan_to_num(out.cpu().numpy(), nan=0.5))
                targets.append(np.nan_to_num(y.numpy(), nan=0.0))
        auc = compute_macro_auc(np.vstack(targets), np.vstack(preds))
        print(f"Fold {fold} Epoch {ep} Val Macro AUC: {auc:.4f}")
        if auc > best_auc:
            best_auc = auc
            torch.save(model.state_dict(), best_path)
    
    print(f"Fold {fold} Best Val Macro AUC: {best_auc:.4f} -> Saved {best_path}")
    fold_aucs.append(best_auc)

print(f"\n=== Wide-ResNet50 Specialist 5-Fold Training Complete! Mean Val Macro AUC: {np.mean(fold_aucs):.4f} ===")
